# CSCI 443 Homework 4

In homework 3, we left off with Z-scores and sampling distributions.  

In this homework we focus on confidence intervals and the effects of
skewness on computing confidence intervals.

**Important Note to Students:**

This homework is designed to help you develop intuition about confidence intervals, sampling distributions, and the behavior of statistical estimators with different sample sizes. These concepts are foundational to understanding statistical inference and quality control applications.

**Please approach these problems without using AI assistance initially.** The purpose of this assignment is to build your understanding through hands-on calculation, coding, and reflection. While AI tools can be helpful for debugging syntax errors or looking up function signatures, over-reliance on AI to solve the problems will defeat the learning objectives and leave you without the intuition needed for exams and real-world applications.

Work through the problems step-by-step, experiment with the code, observe the results, and think critically about what the numbers are telling you. Your future self will thank you for taking the time to truly understand these concepts now.

   
## Part 1: Gaussian Confidence Intervals. 

Assume we have experience with the underlying processes that generate samples and know that the underlying distribution is Gaussian or nearly Gaussian.

**Problem 1.1** You draw the samples in sample set S from this distribution.  Compute a 99% confidence interval using Z scores. 

$$S = \\{5, 3, 9, 2, 5\\}$$

**Answer 1.1**

The purpose of this and subsequent problems in this part is to demonstrate that confidence intervals computed using 
a Gaussian distribution when the sample size is small are inaccurate.

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i = \frac{5 + 3 + 9 + 2 + 5}{5} = \frac{24}{5} = 4.8$$

$$s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

$$s = \sqrt{\frac{(5-4.8)^2 + (3-4.8)^2 + (9-4.8)^2 + (2-4.8)^2 + (5-4.8)^2}{4}}$$

$$s = \sqrt{\frac{0.04 + 3.24 + 17.64 + 7.84 + 0.04}{4}} = \sqrt{\frac{28.8}{4}} = \sqrt{7.2}$$

$$s \approx 2.683$$

The standard error of the mean is:

$$ SE = \frac{s}{\sqrt{n}} $$

$$ SE = \frac{2.683}{\sqrt{5}} \approx \frac{2.683}{2.236} \approx 1.200 $$

The 99\% confidence interval for the mean is:

$$ \bar{x} \pm Z \cdot SE $$

Where $Z$ is the critical value for 99\% confidence ($Z \approx 2.576$):

$$ 4.8 \pm 2.576 \times 1.200 $$

$$ 4.8 \pm 3.091 $$

Confidence interval:

$$ [1.709, 7.891] $$



**Problem 1.2** Write Python code to compute the standard deviation of a sample set using the inverse CDF (quantile function) of the Gaussian distribution. You may use scipy or numpy, but do not use any function that directly computes the confidence interval.  Use this code to confirm the result to problem 1.1.

**Answer 1.2**

In [0]:
import numpy as np
from scipy.stats import norm

# Sample set S
S = np.array([5, 3, 9, 2, 5])
n = len(S)

# Compute sample mean and sample standard deviation
mean = np.mean(S)
stddev = np.std(S, ddof=1)  # sample stddev
print(f"Sample mean: {mean}")
print(f"Sample standard deviation: {stddev}")

# Demonstrate use of inverse CDF (quantile function)
# For a 99% confidence interval, Z = norm.ppf(0.995) ~ 2.575
z99 = norm.ppf(0.995)
print(f"Z score for 99% CI: {z99}")

# Compare stddev result to Problem 1.1
print("Computed stddev matches approx. 2.68 from Problem 1.1")

# compute the confidence interval.
CI = mean + np.array([-1, 1]) * z99 * stddev / np.sqrt(n)
print(f"99% CI: {CI}")  



**Problem 1.3** You built live instrumentation for a production line at a refinery.  For each batch of a refined material, live instrumentation measures n=5 samples, generates a confidence interval, and reports the sample mean of the amount of impurities. To validate the system, you collect a large dataset and divide it into groups of 5 samples. Using all samples in the file `prob1.csv`, plot a histogram of the sampling distribution (i.e., the distribution of sample means from each group of 5).

**Answer 1.3**

In [0]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
prob1_path = f"/Workspace{notebook_dir}/prob1.csv"
print(prob1_path)

df = pd.read_csv(prob1_path)

# Divide data into groups of 5 and compute sample means
# Assuming the CSV has a single column of values
data = df.values.flatten()  # Get all values as a 1D array
print(f"total number of samples in the data {len(data)}")
n_samples = 5
n_groups = len(data) // n_samples  # Number of complete groups

# Reshape data into groups of 5 and compute means
sample_means = []
for i in range(n_groups):
    group = data[i*n_samples:(i+1)*n_samples]
    sample_means.append(np.mean(group))

# Plot histogram of sampling distribution
plt.figure(figsize=(8, 5))
plt.hist(sample_means, bins=20, density=True, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('Sample Mean')
plt.ylabel('Density')
plt.title('Sampling Distribution of Sample Means (n=5)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Number of groups: {n_groups}")
print(f"Mean of sample means: {np.mean(sample_means):.4f}")
print(f"Std of sample means: {np.std(sample_means, ddof=1):.4f}")
print(f"Min of sample means: {np.min(sample_means):.4f}")
print(f"Max of sample means: {np.max(sample_means):.4f}")


**Problem 1.4** Using the full dataset in `prob1.csv`, randomly sample 1000 groups of size n=5 (with replacement). For each group, compute a 99% confidence interval for the mean using a Gaussian approximation (i.e., use the Z-score as in Problems 1.1 and 1.2).

Next, calculate the population mean from the entire dataset in `prob1.csv`.

What fraction of the 1000 confidence intervals contain the ~~true population mean~~ best possible estimate of the population mean, i.e., the sample mean across all data points in the dataset?

Reflect: Is this fraction approximately 99%, as expected from the confidence level? Explain why or why not.

**Answer 1.4**

In [0]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# Load the data
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
prob1_path = f"/Workspace{notebook_dir}/prob1.csv"
df = pd.read_csv(prob1_path)

# Use column if named, otherwise flatten
if 'value' in df.columns:
    data = df['value'].values
else:
    data = df.values.flatten()

# Compute population mean
pop_mean = np.mean(data)

# Settings
n_samples = 5
n_trials = 1000
z = norm.ppf(0.995)   # Z-score for 99% CI (one-sided, for two-sided use 0.995 and 0.005)

contains_true_mean = 0
intervals = []

for _ in range(n_trials):
    sample = np.random.choice(data, n_samples, replace=True)
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    se = sample_std / np.sqrt(n_samples)
    # CI for the mean
    ci_low = sample_mean - z * se
    ci_high = sample_mean + z * se
    intervals.append((ci_low, ci_high))
    # Check if population mean is inside interval
    if ci_low <= pop_mean <= ci_high:
        contains_true_mean += 1

fraction = contains_true_mean / n_trials

print(f"Population mean: {pop_mean:.4f}")
print(f"Fraction of 99% Gaussian CIs containing true mean: {fraction:.3f}")



Only ≈93.9% of the confidence intervals contain the population mean.  
This occurs because the sample standard deviation is biased to the
samples.  It might cause an underestimation of the population standard deviation.
Let's confirm that.

In [0]:
import pandas as pd
import numpy as np

# Load the data
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
prob1_path = f"/Workspace{notebook_dir}/prob1.csv"
df = pd.read_csv(prob1_path)

if 'value' in df.columns:
    data = df['value'].values
else:
    data = df.values.flatten()

# Settings
n_samples = 5
n_trials = 1000

# Collect sample standard deviations
sds = []
for _ in range(n_trials):
    sample = np.random.choice(data, n_samples, replace=True)
    sample_std = np.std(sample, ddof=1)
    sds.append(sample_std)

mean_sample_sd = np.mean(sds)
pop_sd = np.std(data, ddof=1)

print(f"Mean sample standard deviation from bootstrap (n=5): {mean_sample_sd:.4f}")
print(f"Population standard deviation: {pop_sd:.4f}")

if mean_sample_sd < pop_sd:
    print("Mean sample SD underestimates true SD (classic small-sample effect).")
else:
    print("Mean sample SD approximates or exceeds true SD.")

**Problem 1.5** Repeat Problem 1.4, but this time use n=35 samples per group instead of n=5. Randomly sample 1000 groups of size n=35 (with replacement) from `prob1.csv`. For each group, compute a 99% confidence interval for the mean using a Gaussian approximation.

Compute what fraction of the 1000 confidence intervals contain the true population mean.

Reflect: How does this fraction compare to the result from Problem 1.4 (which used n=5)? Why does increasing the sample size improve the coverage rate?

**Answer 1.5**

In [0]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# Load the data
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
prob1_path = f"/Workspace{notebook_dir}/prob1.csv"
df = pd.read_csv(prob1_path)

if 'value' in df.columns:
    data = df['value'].values
else:
    data = df.values.flatten()

# Compute population mean
pop_mean = np.mean(data)

# Settings - now using n=35 instead of n=5
n_samples = 35
n_trials = 1000
z = norm.ppf(0.995)   # Z-score for 99% CI

contains_true_mean = 0
intervals = []

for _ in range(n_trials):
    sample = np.random.choice(data, n_samples, replace=True)
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    se = sample_std / np.sqrt(n_samples)
    # CI for the mean
    ci_low = sample_mean - z * se
    ci_high = sample_mean + z * se
    intervals.append((ci_low, ci_high))
    # Check if population mean is inside interval
    if ci_low <= pop_mean <= ci_high:
        contains_true_mean += 1

fraction = contains_true_mean / n_trials

print(f"Population mean: {pop_mean:.4f}")
print(f"Sample size per group: n={n_samples}")
print(f"Fraction of 99% Gaussian CIs containing true mean: {fraction:.3f}")
print(f"\nComparison:")
print(f"  Problem 1.4 (n=5): ~93.9% coverage")
print(f"  Problem 1.5 (n={n_samples}): {fraction*100:.1f}% coverage")
print(f"\nReflection:")
if fraction > 0.96:
    print(f"With n={n_samples}, the coverage rate is much closer to the theoretical 99%.")
    print("Larger sample sizes reduce the bias in sample standard deviation,")
    print("leading to more accurate confidence intervals and better coverage.")

# Visualize CI widths comparison
plt.hist([ci_high-ci_low for ci_low, ci_high in intervals], bins=20, alpha=0.6, color='green')
plt.xlabel('CI Width')
plt.ylabel('Frequency')
plt.title(f'Distribution of 99% CI Widths (n={n_samples} samples)')
plt.show()

**Problem 1.6**  
Suppose we want our quality control confidence intervals to be empirically accurate: specifically, we want at least 98% of our 99% confidence intervals to contain the ~~true population mean~~ best estimate of the population mean, i.e., the sample mean computed across all samples. For the dataset in `prob1.csv`, determine the minimum sample size `n` (for groups of size `n` drawn with replacement) for which the fraction of 10000 Gaussian 99% confidence intervals containing the true is within 1% of the 99% target confidence (i.e., exceeds 98%).

Present your code, plot the coverage as a function of `n`, and report the minimum `n` that satisfies the requirement.  
Briefly discuss the practical trade-off between statistical confidence and sampling cost.

**Answer 1.6**

In [0]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# Load the data
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
prob1_path = f"/Workspace{notebook_dir}/prob1.csv"
df = pd.read_csv(prob1_path)

if 'value' in df.columns:
    data = df['value'].values
else:
    data = df.values.flatten()

# Compute population mean
pop_mean = np.mean(data)
z = norm.ppf(0.995)  # Z-score for 99% CI

# Test various sample sizes
n_values = range(5, 51, 1)  # Test n from 5 to 50
n_trials = 10000
coverages = []

print("Computing coverage for different sample sizes...")
for n_samples in n_values:
    contains_true_mean = 0
    
    for _ in range(n_trials):
        sample = np.random.choice(data, n_samples, replace=True)
        sample_mean = np.mean(sample)
        sample_std = np.std(sample, ddof=1)
        se = sample_std / np.sqrt(n_samples)
        
        ci_low = sample_mean - z * se
        ci_high = sample_mean + z * se
        
        if ci_low <= pop_mean <= ci_high:
            contains_true_mean += 1
    
    coverage = contains_true_mean / n_trials
    coverages.append(coverage)

# Find minimum n where coverage >= 0.98
min_n = None
for i, n in enumerate(n_values):
    if coverages[i] >= 0.98:
        min_n = n
        break

# Plot coverage as a function of n
plt.figure(figsize=(10, 6))
plt.plot(n_values, coverages, 'o-', linewidth=2, markersize=4, label='Empirical Coverage')
plt.axhline(y=0.99, color='green', linestyle='--', linewidth=2, label='Theoretical 99%')
plt.axhline(y=0.98, color='red', linestyle='--', linewidth=2, label='Target 98%')

if min_n:
    plt.axvline(x=min_n, color='orange', linestyle=':', linewidth=2, label=f'Min n={min_n}')
    plt.plot(min_n, coverages[min_n-5], 'ro', markersize=10)

plt.xlabel('Sample Size (n)', fontsize=12)
plt.ylabel('Coverage (fraction of CIs containing true mean)', fontsize=12)
plt.title('Coverage of 99% Gaussian Confidence Intervals vs Sample Size', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim([0.9, 1.0])
plt.show()

# Report results
print(f"\nPopulation mean: {pop_mean:.4f}")
if min_n:
    print(f"\nMinimum sample size for >=98% coverage: n = {min_n}")
    print(f"Coverage at n={min_n}: {coverages[min_n-5]:.3f}")
else:
    print("\nNo sample size in the tested range achieved >=98% coverage.")


**Problem 1.7**  
A coworker reviews your analysis from Problems 1.4-1.6 and points out that the Gaussian approximation may not be appropriate for small sample sizes. She suggests using the **Student's t-distribution** instead, which accounts for the additional uncertainty introduced when estimating the population standard deviation from small samples.

The t-distribution has heavier tails than the normal distribution, producing wider confidence intervals that better reflect the uncertainty in small-sample estimates. As sample size increases, the t-distribution converges to the normal distribution.

Repeat Problem 1.4 using the **t-distribution** instead of the Gaussian (Z-score) approach:
* Randomly sample 1000 groups of size n=5 from `prob1.csv`
* For each group, compute a 99% confidence interval using the t-distribution with appropriate degrees of freedom
* Calculate what fraction of these intervals contain the ~~true population mean~~ best estimate of the population mean, i.e., the sample mean computed across all samples.

Compare your result to Problem 1.4 (which used the Gaussian approach with n=5 and achieved ~94% coverage). Does the t-distribution produce confidence intervals with coverage closer to the nominal 99%?

**Hint**: Use `scipy.stats.t.ppf()` to get the critical t-value, with degrees of freedom = n-1.

**Answer 1.7**

In [0]:
import pandas as pd
import numpy as np
from scipy.stats import t

# Load the data
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
prob1_path = f"/Workspace{notebook_dir}/prob1.csv"
df = pd.read_csv(prob1_path)

if 'value' in df.columns:
    data = df['value'].values
else:
    data = df.values.flatten()

# Population mean
pop_mean = np.mean(data)

n_samples = 5
n_trials = 1000
contains_true_mean = 0
intervals = []
dofs = n_samples - 1
# Critical value for two-sided 99% CI
t_crit = t.ppf(0.995, dofs)

for _ in range(n_trials):
    sample = np.random.choice(data, n_samples, replace=True)
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    se = sample_std / np.sqrt(n_samples)
    ci_low = sample_mean - t_crit * se
    ci_high = sample_mean + t_crit * se
    intervals.append((ci_low, ci_high))
    if ci_low <= pop_mean <= ci_high:
        contains_true_mean += 1

fraction = contains_true_mean / n_trials

print(f"Population mean: {pop_mean:.4f}")
print(f"Sample size per group: n={n_samples}")
print(f"Fraction of 99% t-distribution CIs containing true mean: {fraction:.3f}")
print("\nCompared to Problem 1.4 (Gaussian):")
print("  Gaussian CI coverage was ~94% \n  t-distribution CI coverage is closer to 99% (nominal)")
if fraction > 0.98:
    print("Using Student's t-distribution corrects for small-sample bias and produces more reliable confidence intervals.")


## Part 2: Formal definition for Skewness

We mostly covered skewness empirically.  "Follow the tails" being a useful mnemonic.

In most cases, a left (negative) skewed distribution is characterized by having
a mean that is less than the median, i.e., the heavy tail on the left has pulled
the mean to the left of the median.  Conversly, a right skewed distribuion is 
characterized by having a mean that is greater than the median.  We will not
encounter any distributions in this class where this is not true.  However, 
the relationship between the mean and median is merely a consequence, it isn't
the formal definition of skewness.  Skewness (\\(\gamma\\)) is most often defined 
as follows

$$\text{Skewness } \gamma = \frac{E[(X-\mu)^3]}{\sigma^3}$$

The sample variance \\(\sigma^2= \frac{1}{n-1}\sum (x_i-\bar{x})^2\\)
uses \\(\frac{1}{n-1}\\) instead of \\(\frac{1}{n}\\) to 
account for sample variance being computed from the same samples
as the sample mean.  

Sample Skewness (\\(G_1\\)) has a somewhat more complicated correction
to handle the bias introduced by computing the sample mean, sample
standard deviation, and sample skewness from the same samples.

$$G_1 = \frac{n}{(n-1)(n-2)}\sum_{i=1}^n \bigg(\frac{x_i-\bar{x}}{s}\bigg)^3$$

where

  - \\(n\\) is the number of samples
  - \\(x_i\\) is the \\(i^{th}\\) sample.
  - \\(\bar{x}\\) is the sample mean.

\\(\frac{n}{(n-1)(n-2)}\\) includes the correction for the bias 
introduced by computing \\(\bar{x}\\), \\(s\\), and 
\\(G_1\\) from the same samples.  The derivation of this
correction is outside the scope of the course, but I welcome
the student to study it further should they wish.




**Problem 2.1** Create code in this notebook to compute the sample skewness.

    def sample_skewness(samples) -> float:
      ...

**Answer 2.1**

In [0]:
import numpy as np

def sample_skewness(samples) -> float:
    n = len(samples)
    if n < 3:
        raise ValueError("sample skewness requires at least 3 samples")

    xbar = np.mean(samples)
    s = np.std(samples, ddof=1)
    shifted = samples - xbar
    sum_cubed = np.sum(shifted ** 3)

    skewness = (
        n / ((n - 1) * (n - 2))
    ) * (sum_cubed / (s ** 3))

    return skewness

**Problem 2.2** Write a unit test in this notebook that confirms that your
implementation of sample skewness returns near 0 \\(\pm 0.05\\) for a
sufficient number of samples drawn from U[0,1].  U[0,1] is symmetric.
All symmetric distributions exhibit 0 skewness.  


**Answer 2.2**

In [0]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Draw sufficient samples from U[0,1]
samples = np.random.rand(10000)

# Compute sample skewness
gamma = sample_skewness(samples)

print(f"Sample skewness from U[0,1]: {gamma:.4f}")
print(f"Expected skewness (symmetric distribution): 0")

# Unit test: assert skewness is within ±0.05 of 0
assert -0.05 < gamma < 0.05, f"Skewness {gamma:.4f} is not within ±0.05 of 0"

print("✓ Test passed: Skewness is approximately 0 for symmetric distribution")

   
**Problem 2.3** Write a second unit test that confirms your implementation correctly detects positive skewness. The exponential distribution with \\(\lambda=1\\) is right-skewed and has a theoretical skewness of 2. Draw a sufficient number of samples from an exponential random variable with \\(\lambda=1\\) and verify that your `sample_skewness` function returns a value within \\(2 \pm 0.05\\), confirming that the distribution exhibits the expected positive (right) skewness.

**Answer 2.3**

In [0]:
import numpy as np

# Set seed for reproducibility
np.random.seed(123)

# Draw samples from exponential distribution with λ=1
# Note: numpy uses scale parameter = 1/λ, so scale=1 means λ=1
samples = np.random.exponential(scale=1, size=100000)

# Compute sample skewness
gamma = sample_skewness(samples)

print(f"Sample skewness from Exp(λ=1): {gamma:.4f}")
print(f"Expected skewness (theoretical): 2.0")

# Unit test: assert skewness is within 2±0.05
assert 2 - 0.05 < gamma < 2 + 0.05, f"Skewness {gamma:.4f} is not within 2±0.05"

print("✓ Test passed: Skewness is approximately 2 for exponential distribution")

**Problem 2.4** Skewness of a distribution is defined as

<!--
\\[ \xcancel{E[(X - \mu)^3] = \int_{-\infty}^{\infty} (x - \mu)^3 f(x) \, dx} \\]

*Revised*
-->

\\[ \frac{E[(X - \mu)^3]}{\sigma^3} = \int_{-\infty}^{\infty} \frac{(x - \mu)^3}{\sigma^3} f(x) \, dx \\]

A triangular distribution is defined by three parameters.  It has the pdf

\\[
f(x) = \begin{cases} 
  0      & \text{if } x \leq a \\
  \frac{2\cdot(x-a)}{(b-a)(c-a)}& \text{if } a \leq x \leq c \\
  \frac{2\cdot(c-x)}{(b-a)(c-b)}& \text{if } c \leq x \leq b \\
  0 & \text{if } x \geq b \\
\end{cases}
\\]

where 

 * \\(a\\) is the left corner of the triangle, i.e., the lower bound
   of the distribution's pdf.   
 * \\(b\\) is the right corner of the triangle, i.e., the upper bound
   of the distribution's pdf.
 * \\(c\\) is at the peak of the triangle, i.e., the mode of the
   distribution's pdf.

An example triangular distribution is shown below.


In [0]:
import numpy as np
import matplotlib.pyplot as plt

def plot_triangular_distribution(a, b, c):
    # Generate x values across the support
    x = np.linspace(a, b, 500)
    y = np.zeros_like(x)
    
    # Compute the pdf according to the triangular definition
    for i, xi in enumerate(x):
        if a <= xi <= c:
            y[i] = 2 * (xi - a) / ((b - a) * (c - a))
        elif c < xi <= b:
            y[i] = 2 * (b - xi) / ((b - a) * (b - c))
        # else remain 0
    
    plt.plot(x, y, lw=2, color='navy')
    plt.fill_between(x, y, color='skyblue', alpha=0.5)
    plt.xlabel('x')
    plt.ylabel('Density')
    plt.title(f'Triangular Distribution PDF\n a={a}, b={b}, c={c}')
    plt.grid(True, alpha=0.3)
    plt.show()

In [0]:
plot_triangular_distribution(a=0, b=1, c=2/3)


The mean of a triangular distribution is given by 

\\[ \mu = \frac{a + b + c}{3}\\]

A right triangular distribution with the peak on the right edge of the 
triangle has \\(b=c\\), as shown below.  The triangle is shifted to the 
left so that the mean resides at 0.


In [0]:
a, b, c = -2/3, 1/3, 1/3 
plot_triangular_distribution(a, b, c)

PDF of a right triangle distribution:

\\[ f(x) = \frac{2(x-a)}{(b-a)^2} \quad \text{for } a \leq x \leq b \\]

However, \\(b-a=1\\), \\(a=-2/3\\), and \\(b=1/3\\) causing the above to simplify

\\[ f(x) = 2(x+\frac{2}{3}) \quad \text{for } -\frac{2}{3} \leq x \leq \frac{1}{3} \\]

Derive the skewness for this distribution.

**Answer 2.4:**

\\[ \frac{E[(X - \mu)^3]}{\sigma^3} = \frac{1}{\sigma^3} \int_{-\frac{2}{3}}^{\frac{1}{3}} x^3 \cdot 2(x+\frac{2}{3}) \, dx \\]

$$\frac{1}{\sigma^3} \bigg(\frac{2}{5}x^5 + \frac{1}{3}x^4 \bigg)\bigg|_{x=-\frac{2}{3}}^{\frac{1}{3}}$$

$$\frac{1}{\sigma^3} \bigg(\frac{2}{5}(\frac{1}{3})^5 + \frac{1}{3}(\frac{1}{3})^4 - \frac{2}{5}(-\frac{2}{3})^5 - \frac{1}{3}(-\frac{2}{3})^4 \bigg)$$

$$\gamma = \frac{-0.0\overline{074}}{\sigma^3}$$

The standard deviation of a triangular distribution is given by

\\[ \sigma = \sqrt{\frac{a^2 + b^2 + c^2 - ab - ac - bc}{18}}\\]

Because \\(b==c\\) the standard deviation simplifies to

<!-- \\[ \sigma = \sqrt{\frac{a^2 + b^2 + \cancel{b^2} - ab - ab - \cancel{b^2}}{18}}\\]
-->
\\[ \sigma = \sqrt{\frac{a^2 + b^2 + b^2 - ab - ab - b^2}{18}}\\]

\\[ \sigma = \frac{\sqrt{a^2 - 2ab + b^2}}{3\sqrt{2}} = \frac{|a-b|}{3\sqrt{2}} = \frac{1}{3\sqrt{2}} \\]

\\[ \sigma \approx 0.2357 \\]

\\[ \gamma = (3\sqrt{2})^3 \cdot -0.0\overline{074} \\]

\\[ \boxed{\gamma \approx -0.566} \\]


## Part 3: Some intuition behind skewness

With variance we look at the sum of the squares 
\\(\sigma^2= \frac{1}{n-1}\sum (x_i-\bar{x})^2\\).
The square means that all terms in the summation are positive.  The 
further samples are away from the mean in either direction their
impact on the variance increases with the square of that distance.
Skewness uses the sum of the cubes.  Consider the case when
\\(\bar{x}\\) is 0 and let \\(C\\) denote \\(n/((n-1)(n-2))\\). The
computation of sample skewness simplifes to a sum of cubics: 

$$C\cdot \sum_{i=0}^{n} x^3$$

\\(x^3\\) is an odd function and as such any sample which is to the left of the
mean by \\(d_i = x_i-\bar{x}\\) will cancel with a sample the same distance on the
right side \\(d_j = x_j-\bar{x}\\) if \\(d_i = -d_j\\).  A symetric distribution 
thus has 0 skewness regardles of its variance.


**Problem 3.1** Write code in this notebook to draw 100 samples from a uniform random
variable \\(U[0,1]\\). Use matplotlib to create a relative
frequency histogram samples with 5 bins to confirm that the shape is 
roughly uniform.  Not all buckets will obtain the same number of samples.   
Aside: We will go over tests for measuring how close samples match a 
hypothesized distribution, for now a visual assessment is adequate.


In [0]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec

# Generating two separate plots: one for the histogram and one for
# the samples along the real line
np.random.seed(23)

n = 10000   # number of samples

# Generate n samples obeying U[-1, 1]
samples = np.random.uniform(-1, 1, n)

# creating two subplots where the second (samples on a real-line) has less height.
fig = plt.figure(figsize=(6, 5))

# This creates a grid of 2 rows with a 7:1 height ratio
gs = gridspec.GridSpec(2, 1, height_ratios=[7, 1], hspace=0)

# add_subplot returns an Axes object.  Each Axes object
# is essentially a subplot.  It has axes, a title, data, etc.
axs = []
axs.append(fig.add_subplot(gs[0]))
axs.append(fig.add_subplot(gs[1]))

# Plot the histogram of the samples
axs[0].hist(samples, bins=5, density=True, alpha=0.5, color='orange')
axs[0].set_title('Histogram from U[-1, 1]')
axs[0].set_xticks([]) # Hiding the x-axis
axs[0].set_ylabel('Density')

# Plotting the samples along the real line
axs[1].scatter(samples, np.zeros_like(samples), alpha=0.5, 
               marker='|', color='blue')
axs[1].set_yticks([]) # Hiding the y-axis

# Setting the same x limits for both plots for easy comparison
for ax in axs:
    ax.set_xlim([-1.2, 1.2])

plt.tight_layout()
plt.show()


**Problem 3.2** Create a plot that shows the sum of the squares 
\\(\sum_{i=1}^n (x_i-\bar{x})^2 \\) for the same samples as generated in Problem 3.1.
The x-axis should be \\(i\\) where \\(i\\) denotes the \\(i^{th}\\) sample.
The y-axis should be the sum of the squares up to the \\(i^{th}\\) sample.
This plot should be non-decreasing.  It demonstrates how sum of squares increases
whether samples fall above or below the mean.  

In [0]:
# Using samples from above.
diffs = samples - np.mean(samples)
sqx = np.multiply(diffs, diffs)  # performs elementwide multiplicatoin (Hadamard product) between diffs with itself.
sumsqx = np.cumsum(sqx)
plt.scatter(range(len(samples)), sumsqx)
plt.title("cumulative sum of $(x_i-\\overline{x})^2$")
plt.xlabel("i")
plt.ylabel("sum")
plt.show()

**Problem 3.3**  Order the samples from Problem 3.1 from smallest to largest and create another plot.  


In [0]:
# Using samples from 3.1 above.
sorted_samples = sorted(samples)
diffs = sorted_samples - np.mean(sorted_samples)
sqx = np.multiply(diffs, diffs)  # performs elementwide multiplicatoin (Hadamard product) between diffs with itself.
sumsqx = np.cumsum(sqx)
plt.scatter(range(len(samples)), sumsqx)
plt.title("cumulative sum of $(x_i-\\overline{x})^2$")
plt.xlabel("i")
plt.ylabel("sum")
plt.show()

**Problem 3.4** When the samples are ordered, what happens? Explain the shape. What does the shape of the curve tell us about the contribution to the variance of points that are farther from the mean?

**Answer 3.4**: The shape is an increasing function that grows steeper for the extrema.  Given that
the underlying distribution is uniform we would expect the median to be very near the
mean, as such the middle of the plot should be at or near \\(\bar{x}\\) and all those
to the left are progressively farther from the mean as we go left.  Conversely, all
those to the right of middle of the the plot should be above the mean and progressively
as we go to the right.  This demonstrates that those farther from the mean contribute
more strongly to the variance.  This is consistent with summing the squares of the 
distance from the mean.

**Problem 3.5** Create another plot that shows the sum of the 
cubes \\(\sum_{i=1}^n (x_i - \bar{x})^3\\) for the same samples 
generated in Problem 3.1.  The resulting plot should NOT be
increasing.

**Answer 3.5**

In [0]:
# Using samples from above.
print(f"mean of the samples={np.mean(samples)}")
xbar = np.mean(samples)
#xbar = 0
diffs = samples - xbar
cubx = np.multiply(np.multiply(diffs, diffs), diffs) 
sumcubx = np.cumsum(cubx)
plt.scatter(range(len(samples)), sumcubx)
plt.title("cumulative sum of $(x_i-\overline{x})^3$")
plt.xlabel("i")
plt.ylabel("sum of cubes")

plt.show()

The plot above is trending upward but it is not increasing.  It looks like a random
walk.  Compared to the range that is seen for the y-axis in the plot sum square
of the distances of \\(x\\) from the mean, the range for this plot is 
small.  This makes sense given that the \\((x_i - \bar{x})^3\\) is odd.  As a result,
the values to the right of the mean should largely cancel the values to the left of hte
mean.  For symmetric distributions, the sum of the cubes should stay near 0.

**Problem 3.6** Order the samples from smallest to largest and create the same plot again
but with the ordered samples.  


**Answer 3.6**

In [0]:
#S = np.array([1, 2, 3])
#cubs = np.multiply(np.multiply(S, S), S)
#print(f"S={S}, cubs = {cubs}")

sorted_samples = np.sort(samples)
diffs = sorted_samples - np.mean(sorted_samples)
print(f"np.mean after subtracting the mean is {np.mean(diffs)}")
cubx = np.multiply(np.multiply(diffs, diffs), diffs) 
sumcubx = np.cumsum(cubx)
plt.scatter(range(len(samples)), sumcubx)
plt.title("cumulative sum of $(x_i-\overline{x})^3$")
plt.xlabel("i")
plt.ylabel("sum of cubes")

plt.show()


**Problem 3.7** How does this plot differ from the plot for cumulative sum of squares?
Note that the function is no longer non-decreasing.  How does the shape of the 
curve affect the contribution of the samples 
to the left of the mean vs. to the right of the mean for 
an unskewed distribution? 

**Answer 3.7**

* Those to the left of the mean drove the sum of cubes negative.
  Those to the right of the mean push the sum back toward zero.
  For an unskewed distribution, I would expect those to the left of the 
  mean to roughly cancel those to the right of the mean ending up with
  a near-zero skewness.



**Problem 3.8** How does the shape of this curve affect the contribution of samples 
farther from the mean than nearer to the mean? What happens if more samples
are near the mean on one-side of the distribution as would occur with 
a skewed distribution?

**Answer 3.8**

* Since the difference of each sample from the mean is cubed, bigger differences have far greater impact.  Because \\(y=x^3\\) is an odd function, negative differences become substantially more negative, and conversely, positive differences become substantially more positive.  If the distribution is skewed then one side likely has a heavier tail.  The points in the tail have substantially more affect on the skewness than those near the mean and as such skewness is positive when a distribution is heavier right tail and skewness is negative when a distribution has a heavier left tail.


## Part 4: Confidence Intervals for Instrument Data Obeying an Unknown Disribution

You are designing instrumentation for a physical system and the collected data does not follow a known distribution. The sample data is provided in the file `prob4.csv` in the same directory as this notebook.



**Problem 4.1**
Load the data from `prob4.csv` and plot a histogram of its values. Carefully observe the shape of the distribution. Is it symmetric or skewed? Record your observations and discuss any features that stand out.

**Answer 4.1**

In [0]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
prob4_path = f"/Workspace{notebook_dir}/prob4.csv"

# Load data
try:
    df = pd.read_csv(prob4_path)
    if 'value' in df.columns:
        data = df['value'].values
    else:
        data = df.values.flatten()
except Exception as e:
    print("Error loading prob4.csv:", e)
    data = np.array([])

print(f"Total number of samples: {len(data)}")

# Plot histogram
if data.size > 0:
    plt.figure(figsize=(8,5))
    plt.hist(data, bins=30, density=True, color='skyblue', edgecolor='black', alpha=0.7)
    plt.xlabel('Sample Value')
    plt.ylabel('Density')
    plt.title('Histogram of Instrument Data (prob4.csv)')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Print basic stats
    print(f"Min: {np.min(data):.3f}, Max: {np.max(data):.3f}, Mean: {np.mean(data):.3f}, Median: {np.median(data):.3f}")
    print(f"Std: {np.std(data):.3f}")
else:
    print("No data loaded for prob4.csv.")

The distribution is heavily right skewed.

**Problem 4.2**
Compute the skewness of the data in `prob4.csv` using your `sample_skewness` function from earlier in this homework. Report the computed skewness and comment on what it indicates about the shape of the distribution.

**Answer 4.2**

In [0]:
import pandas as pd
import numpy as np

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
prob4_path = f"/Workspace{notebook_dir}/prob4.csv"

df = pd.read_csv(prob4_path)
if 'value' in df.columns:
    data = df['value'].values
else:
    data = df.values.flatten()

# Compute skewness
try:
    gamma = sample_skewness(data)
except Exception as e:
    gamma = None
    print("Error computing skewness:", e)

if gamma is not None:
    print(f"Sample skewness (prob4.csv): {gamma:.4f}")
    if gamma > 0:
        print("The distribution is right-skewed, with a heavy right tail.")
    elif gamma < 0:
        print("The distribution is left-skewed, with a heavy left tail.")
    else:
        print("The distribution is symmetric.")

**Problem 4.3**
For n=4, randomly sample 10000 groups from the data in `prob4.csv`. Calculate a 99% confidence interval for the mean of each group using the t-distribution. What fraction of the intervals contain the ~~true population mean~~ best estimate of the population mean, i.e., the sample mean computed across all samples? How many produce impossible (negative) interval values for the mean?

**Answer 4.3**

In [0]:
import pandas as pd
import numpy as np
from scipy.stats import t

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
prob4_path = f"/Workspace{notebook_dir}/prob4.csv"

df = pd.read_csv(prob4_path)
if 'value' in df.columns:
    data = df['value'].values
else:
    data = df.values.flatten()

n_samples = 4
n_trials = 10000
pop_mean = np.mean(data)
t_crit = t.ppf(0.995, n_samples-1)
containing = 0
neg_bounds = 0
intervals = []

for _ in range(n_trials):
    group = np.random.choice(data, n_samples, replace=True)
    xbar = np.mean(group)
    s = np.std(group, ddof=1)
    se = s / np.sqrt(n_samples)
    ci_lo = xbar - t_crit * se
    ci_hi = xbar + t_crit * se
    intervals.append((ci_lo, ci_hi))
    if ci_lo <= pop_mean <= ci_hi:
        containing += 1
    if ci_lo < 0:
        neg_bounds += 1

percent_contain = containing / n_trials * 100
percent_neg = neg_bounds / n_trials * 100

print(f"Population mean = {pop_mean:.4f}")
print(f"Fraction of 99% CIs containing mean: {percent_contain:.2f}%")
print(f"Fraction with negative lower bound: {percent_neg:.2f}%")
print("Example confidence intervals:")
for i, (lo, hi) in enumerate(intervals[:5]):
    print(f"CI {i+1}: [{lo:.3f}, {hi:.3f}]")

**Problem 4.4**
Discuss: Based on your results, does the t-distribution approach provide reliable coverage and sensible interval values for this data? Why or why not?  What percentage of intervals contain the sample mean?

**Answer 4.4**

The t-distribution approach does **not** provide reliable coverage or sensible interval values for this data. There are two major problems:

1. **Under-coverage**: Only approximately 92% of the supposedly 99% confidence intervals contain the best estimate of the population mean. This means we are getting significantly less coverage than the nominal 99% we expect. The t-distribution assumes the underlying data is approximately normal, but our heavily right-skewed data violates this assumption.

2. **Impossible negative values**: Approximately 94% of the confidence intervals have a negative lower bound. This is physically meaningless for this dataset, which represents measurements from a physical process that can only produce positive values (the distribution has non-zero probability density only for positive values). The fact that nearly all our confidence intervals include impossible negative values demonstrates that the t-distribution approach is fundamentally inappropriate for this data.

The t-distribution method fails here because it assumes symmetry around the mean. For heavily skewed data with strictly positive values, we need a method that respects the natural constraints of the data—this is where bootstrapping becomes valuable.

**Problem 4.5**
Use bootstrapping to empirically determine 99% confidence intervals for the sample mean (with n=4 samples per group). Generate 10000 confidence intervals using this approach.  What percentage contain the sample mean?  Compare the bootstrap intervals to those produced by the t-distribution.

**Answer 4.5**

This is admittedly a confusing question.   It doesn't explicitly say how many resamples to use to create the bootstrap sampling distribution for each confidence interval.   Because we are using n=4 samples, there are only 

$$\binom{n+k-1}{k}$$ 

where *k* is the number of samples in the resample and $n$ is the number of samples to draw from with replacement.   We would normally set *n=k* meaning there are 

$$\binom{n+n-1}{n} = \frac{(2n-1)!}{n!(2n-1-n)!} = \frac{(2n-1)!}{n!(n-1)!} = \frac{7!}{4! 3!} = 35$$

possible sample sets. If my original set is [1,2,3,4] then my resamples might look like [2,1,1,4], [4,4,1,1], ...
That means there are at most 35 distinct sample means. 

We could genereate 10000 resamples to construct a bootstrap sampling distribution, but it would be massive overkill.  Unfortunately the problem doesn't say how many resamples to use!

Even though 10000 is overkill, it is the closest to a rational interpretation of this problem, so let's do it.



In [0]:
n = 4  # number of samples to pull
R = 10000  # number of bootstrap resamples

import numpy as np

def bootstrap_mean_distributions(data, n, M, R, rng=None):
    """
    Create M bootstrap sample distributions for the sample mean.

    Parameters
    ----------
    data : np.ndarray
        1-D array representing the underlying observed dataset / population proxy.
    n : int
        Size of each original sample drawn from `data`.
    M : int
        Number of bootstrap distributions to create.
    R : int
        Number of bootstrap resamples within each bootstrap distribution.
    rng : np.random.Generator or None
        Random number generator. If None, a default generator is created.

    Returns
    -------
    original_samples : np.ndarray
        Shape (M, n). Each row is one original sample of size n drawn from `data`.
    bootstrap_means : np.ndarray
        Shape (M, R). Row i contains the R bootstrap sample means derived
        from original_samples[i].
    """
    data = np.asarray(data, dtype=float)
    if data.ndim != 1:
        raise ValueError("data must be a 1-D numpy array")
    if n <= 0 or M <= 0 or R <= 0:
        raise ValueError("n, M, and R must all be positive")

    if rng is None:
        rng = np.random.default_rng()

    N = len(data)

    # Step 1: draw M original samples of size n from the underlying data
    sample_idx = rng.integers(0, N, size=(M, n))
    original_samples = data[sample_idx]   # shape: (M, n)

    # Step 2: for each original sample, draw R bootstrap resamples of size n
    # Each resample is drawn with replacement from that sample's n values.
    resample_idx = rng.integers(0, n, size=(M, R, n))
    bootstrap_resamples = np.take_along_axis(
        original_samples[:, None, :],   # shape: (M, 1, n)
        resample_idx,                   # shape: (M, R, n)
        axis=2
    )

    # Step 3: compute mean of each bootstrap resample
    bootstrap_means = bootstrap_resamples.mean(axis=2)  # shape: (M, R)

    return original_samples, bootstrap_means




In [0]:
M = 10000
R = 200    # I reduced this from 10000 to speed it up, but it doesn't appreciably change the result.
n=4

rng = np.random.default_rng(443)

original_samples, bootstrap_means = bootstrap_mean_distributions(data, n, M, R, rng)

print("samples shape:", samples.shape)   # 1-d of length N
# (M, n)
print("bootstrap_means shape:", bootstrap_means.shape)     # (M, R)

# Example: percentile bootstrap CI for the first original sample
print("First five 99% bootstrap CIs")
for i in range(5):
  ci0 = np.percentile(bootstrap_means[i], [0.5, 99.5])
  print("99% bootstrap CI:", ci0)

# Example: fraction of M intervals containing the overall mean of data
target_mean = data.mean()
cis = np.percentile(bootstrap_means, [0.5, 99.5], axis=1).T  # shape: (M, 2)
coverage = np.mean((cis[:, 0] <= target_mean) & (target_mean <= cis[:, 1]))

print("Coverage vs overall mean:", coverage)

# Example: fraction with negative lower bound
frac_negative_lower = np.mean(cis[:, 0] < 0)
print("Fraction with negative lower bound:", frac_negative_lower)


This demonstrates that 4 samples per CI is simply not enough to create a CI that
achieves the desired confidence using bootstrapping on a highly skewed distribution.
Let's look at a single bootstrap sample to see what happens.



In [0]:

# Find unique possible means and their frequencies
unique_means, counts = np.unique(bootstrap_means[0], return_counts=True)
pmf = counts / counts.sum()

plt.bar(unique_means, pmf, width=0.02)
plt.xlabel('Bootstrap Sample Mean')
plt.ylabel('Probability')
plt.title('PMF of Bootstrap Sample Means (n=4)')
plt.show()


When we try to create a bootstrap distribution to approximate the sampling distribution of the sample mean using *n=4* samples per resample, we end up with a PMF with only few means with non-zero probability.  *n=4* is not enough to characterize the sampling distribution of the sample mean.


**Problem 4.6**

Monte Carlo simulation is a computational technique used to approximate the properties of a distribution by randomly sampling from observed data. In this context, you will use Monte Carlo simulation to generate an empirical sampling distribution for the sample mean of n=4 for the instrument data in prob4.csv. 

- Randomly sample many groups of n=4 from the dataset.
- Compute the mean for each group.
- Plot a histogram of the resulting distribution of sample means.

How could you use this empirical sampling distribution to construct confidence intervals for the sample mean?



In [0]:
import numpy as np

def empirical_mean_distribution(data, n, R, rng=None):
    """
    Create an empirical sampling distribution for the sample mean
    by repeatedly drawing samples of size n from `data`.

    Parameters
    ----------
    data : np.ndarray
        1-D array representing the underlying dataset / population proxy.
    n : int
        Size of each sample drawn from `data`.
    R : int
        Number of samples to draw; this is the size of the empirical
        sampling distribution.
    rng : np.random.Generator or None
        Random number generator. If None, a default generator is created.

    Returns
    -------
    samples : np.ndarray
        Shape (R, n). Each row is one sample of size n drawn from `data`.
    sample_means : np.ndarray
        Shape (R,). The mean of each sample.
    """
    data = np.asarray(data, dtype=float)
    if data.ndim != 1:
        raise ValueError("data must be a 1-D numpy array")
    if n <= 0 or R <= 0:
        raise ValueError("n and R must be positive")

    if rng is None:
        rng = np.random.default_rng()

    N = len(data)

    # Draw R samples of size n from the population proxy
    idx = rng.integers(0, N, size=(R, n))
    samples = data[idx]
    sample_means = samples.mean(axis=1)

    return samples, sample_means



In [0]:

n = 4
R = 10000

rng = np.random.default_rng(443)

samples, sample_means = empirical_mean_distribution(data, n=n, R=R, rng=rng)

# Plot histogram of the empirical sampling distribution of the sample mean
plt.figure(figsize=(8, 5))
plt.hist(sample_means, bins=50, edgecolor='black')
plt.xlabel("Sample mean")
plt.ylabel("Frequency")
plt.title(f"Empirical Sampling Distribution of the Sample Mean (n={n}, R={R})")
plt.grid(alpha=0.3)
plt.show()



If we treat the full dataset as a proxy for the population, we can go one step further:

1. Let 
  
  $$\hat{\mu}$$
  
   be the mean of the full dataset
	
2. Use simulation to approximate the distribution of

$$\bar{X} - \hat{\mu}$$

3. Construct an interval a lower error 

  $$e_L = q_{0.005}- \hat{\mu}$$

and

  $$e_U = q_{0.995} - \hat{\mu}$$

where q is a suitable quantile from the empirical mean distribution earlier in Problem 4.6.

4. For a sample mean consruct CI:


$$[\bar{x} + e_L, \bar{x} + e_U]$$

I added $e_L$ because it is negative the way I defined it unless the distribution is so left skewed that the mean falls below the 0.5% percentile.  Similarly $e_U$ is positive unless the
distribution is so right skewed that the mean is higher than the 99.5% percentile.

This is a simulation-based way to approximate a confidence interval.

**Problem 4.7**
Now, use your empirical distribution (from Problem 4.6) to construct 10,000 confidence intervals for n=4. For example, you may use the 0.5th and 99.5th percentiles of the empirical distribution as the interval bounds. Compute what percentage of these intervals contain the ~~population mean~~best estimate of the population mean, i.e., the sample mean across all samples in the entire dataset, and how many intervals have a negative lower end. Compare your results to the intervals constructed using the t-distribution in Problem 4.3.

In [0]:
M = 1000
R = 500    # I reduced this from 10000 to speed it up, but it doesn't appreciably change the result.
n=100

original_samples, bootstrap_means = bootstrap_mean_distributions(data, n, M, R)

print("samples shape:", samples.shape)   # 1-d of length N
# (M, n)
print("bootstrap_means shape:", bootstrap_means.shape)     # (M, R)

# Example: percentile bootstrap CI for the first original sample
print("First five 99% bootstrap CIs")
for i in range(5):
  ci0 = np.percentile(bootstrap_means[i], [0.5, 99.5])
  print("99% bootstrap CI:", ci0)

# Example: fraction of M intervals containing the overall mean of data
target_mean = data.mean()
cis = np.percentile(bootstrap_means, [0.5, 99.5], axis=1).T  # shape: (M, 2)
coverage = np.mean((cis[:, 0] <= target_mean) & (target_mean <= cis[:, 1]))

print("Coverage vs overall mean:", coverage)

# Example: fraction with negative lower bound
frac_negative_lower = np.mean(cis[:, 0] < 0)
print("Fraction with negative lower bound:", frac_negative_lower)


**Answer 4.7**. For this to make sense we should use the procedure I mentioned in the proposed answer for 4.6 where we simulate X-\mu.  



In [0]:
import numpy as np

# data          -> 1-D numpy array containing the full dataset (from Problem 4.1)
# sample_means  -> empirical sampling distribution of the mean for n=4
#                  produced in Problem 4.6

rng = np.random.default_rng(443)

# Best estimate of the population mean from the full dataset
mu_hat = data.mean()

# -------------------------------------------------------------------
# Step 1: Convert the empirical sampling distribution into an
#         empirical error distribution by centering at mu_hat.
# -------------------------------------------------------------------
errors = sample_means - mu_hat

# Lower and upper 99% quantiles of the error distribution
e_L, e_U = np.percentile(errors, [0.5, 99.5])

print(f"Estimated population mean (mu_hat): {mu_hat:.4f}")
print(f"e_L = {e_L:.4f}")
print(f"e_U = {e_U:.4f}")

# -------------------------------------------------------------------
# Step 2: Draw 10,000 fresh samples of size n=4 from data.
#         For each sample, compute xbar.
# -------------------------------------------------------------------
M = 10000
n = 4

idx = rng.integers(0, len(data), size=(M, n))
samples = data[idx]
xbars = samples.mean(axis=1)

# -------------------------------------------------------------------
# Step 3: Construct an empirical 99% CI for each observed xbar.
#
# Using the error quantiles:
#   CI = [xbar + e_L, xbar + e_U]
#
#   P(e_L <= Xbar - mu <= e_U) ≈ 0.99
#
# -------------------------------------------------------------------
lower = xbars + e_L
upper = xbars + e_U

# -------------------------------------------------------------------
# Step 4: Evaluate the intervals
# -------------------------------------------------------------------
contains_mu_hat = (lower <= mu_hat) & (mu_hat <= upper)
coverage_pct = 100 * contains_mu_hat.mean()

negative_lower_count = np.sum(lower < 0)
negative_lower_pct = 100 * np.mean(lower < 0)

print(f"Percentage of intervals containing mu_hat: {coverage_pct:.2f}%")
print(f"Number of intervals with negative lower bound: {negative_lower_count}")
print(f"Percentage with negative lower bound: {negative_lower_pct:.2f}%")

What this reveals is that with very skewed distributions *n=4* is simply not enough to create an accurate CI.

**Problem 4.8**
Suppose, due to cost constraints, you are only able to collect the first 100 data points from `prob4.csv` (rather than the full dataset). Using classical bootstrapping:

- Draw 10,000 bootstrap samples, each of size n=100 (sampled with replacement from these 100 points).
- For each bootstrap sample, compute the sample mean.
- Plot a histogram of the resulting bootstrap sample means.
- Use the bootstrap distribution to construct a 99% confidence interval for the mean.


In [0]:
import numpy as np
import matplotlib.pyplot as plt

# Assume `data` is already loaded from prob4.csv as a 1-D numpy array
rng = np.random.default_rng(443)

# Step 1: Use only the first 100 data points
sample = data[:100]
n = len(sample)

# Step 2: Generate 10,000 bootstrap samples (with replacement)
R = 10000
idx = rng.integers(0, n, size=(R, n))
bootstrap_samples = sample[idx]

# Step 3: Compute the sample mean for each bootstrap sample
bootstrap_means = bootstrap_samples.mean(axis=1)

# Step 4: Plot histogram of bootstrap means
plt.figure(figsize=(8, 5))
plt.hist(bootstrap_means, bins=50, edgecolor='black')
plt.xlabel("Bootstrap Sample Mean")
plt.ylabel("Frequency")
plt.title("Bootstrap Distribution of the Sample Mean (n=100)")
plt.grid(alpha=0.3)
plt.show()

# Step 5: Construct 99% confidence interval using percentiles
ci_lower, ci_upper = np.percentile(bootstrap_means, [0.5, 99.5])

print(f"99% Bootstrap Confidence Interval for the Mean: [{ci_lower:.4f}, {ci_upper:.4f}]")


**Problem 4.9**
Using the bootstrap sampling distribution of the mean generated in Problem 4.8 (from the first 100 data points in `prob4.csv`), generate 1000 confidence intervals for the sample mean, each based on a group of n=4 randomly sampled values from the full dataset in `prob4.csv`.

For each confidence interval:
- Use the bootstrap distribution to determine the interval bounds (for example, the 0.5th and 99.5th percentiles of the bootstrap means).

What percentage of these intervals contain the true population mean (computed from all data in `prob4.csv`)?



**Answer 4.9** Don't grade this problem.  This problem, as written, is not correct. The bootstrap distribution constructed in Problem 4.8 is based on samples of size n = 100, and therefore approximates the sampling distribution of the mean for samples of size 100. It cannot be used to construct confidence intervals for samples of size n = 4, since the variability and shape of the sampling distribution depend strongly on the sample size.

Instead of answering the question as asked, let's consider bootstrapping with samples of 100 samples drawn from the file `prob4.csv`.

A correct bootstrap procedure would be:
1. Draw a sample of size 100 from the full dataset in prob4.csv.
2. Treat this sample as the observed dataset.
3. Generate many bootstrap resamples (with replacement) of size 100 from this sample.
4. Compute the sample mean for each resample.
5. Use the resulting bootstrap distribution to construct a confidence interval for the mean.

To evaluate performance, this process can be repeated multiple times (e.g., 1000 times), each time starting with a new sample of size 100, and checking how often the resulting intervals contain the true mean of the full dataset.

In [0]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
n = 100        # sample size for each observed sample
M = 1000       # number of confidence intervals to generate
R = 10000      # number of bootstrap resamples inside each interval

# Treat the mean of the full dataset as the "true" population mean
true_mean = data.mean()

# Storage for interval endpoints
ci_lower = np.empty(M)
ci_upper = np.empty(M)

# ------------------------------------------------------------
# Repeated bootstrap experiment
# ------------------------------------------------------------
for i in range(M):
    # Step 1: draw one observed sample of size n from the full dataset
    observed_idx = rng.integers(0, len(data), size=n)
    observed_sample = data[observed_idx]

    # Step 2: generate R bootstrap resamples of size n from this sample
    bootstrap_idx = rng.integers(0, n, size=(R, n))
    bootstrap_samples = observed_sample[bootstrap_idx]

    # Step 3: compute bootstrap means
    bootstrap_means = bootstrap_samples.mean(axis=1)

    # Step 4: percentile bootstrap 99% confidence interval
    ci_lower[i], ci_upper[i] = np.percentile(bootstrap_means, [0.5, 99.5])

# ------------------------------------------------------------
# Evaluate coverage
# ------------------------------------------------------------
contains_true_mean = (ci_lower <= true_mean) & (true_mean <= ci_upper)
coverage_count = np.sum(contains_true_mean)
coverage_pct = 100 * np.mean(contains_true_mean)

negative_lower_count = np.sum(ci_lower < 0)
negative_lower_pct = 100 * np.mean(ci_lower < 0)

print("\nFirst five intervals:")
for i in range(5):
    print(f"[{ci_lower[i]:.6f}, {ci_upper[i]:.6f}]")
    
print(f"True mean from full dataset: {true_mean:.6f}")
print(f"Number of 99% bootstrap intervals: {M}")
print(f"Coverage percentage: {coverage_pct:.2f}%")
print(f"Percentage with negative lower bound: {negative_lower_pct:.2f}%")




**Problem 4.10**
Is 100 samples enough for reasonable bootstrapping? Compare your confidence interval to that obtained with the full dataset (Problems 4.6–4.7). Discuss any limitations or potential biases that may arise with only 100 samples.



Using *n=100*, the coverage is much closer to the 99% that we want compared to bootstrapping with *n=4*, but it still isn't 99%.   It underscores the difficulty of dealing with highly skewed distributions.

Creating an empirical sampling distribution from the entire dataset as in Problem 4.6 gives us a much better approximation of the sampling distribution for *n=4* but it doesn't solve the inherent problem of using only 4 samples per sample mean.

Problem 4.7 tries to reuse the empirically derived sampling distribution for *n=4* but it does so by shifting the empirical sampling distribution by the sample mean.  Tihs doesn't solve the problem of using only 4 samples per sample mean.


